# Lesson 13b: Alignment — RLHF and Preference Optimisation — Practical

13a derived the Bradley-Terry reward model and DPO on synthetic scalar
items and a toy two-response policy. This notebook runs the same ideas
on an actual language model: a character-level LSTM (7b's architecture)
is pretrained on text, a small reward model is trained on a tiny
preference set built from it, DPO is run directly on the language
model's own weights, and the result is compared before and after —
including one honestly reproduced failure mode.

By the end of this notebook you will have:
- **trained a small reward model** on a tiny preference set and measured
  its held-out pairwise accuracy,
- **run DPO on a real character-level language model** and compared its
  generations on held-out prompts before and after,
- and **reproduced a genuine reward-hacking failure**: pushed alignment
  training far enough that the model games the preference signal instead
  of genuinely improving.

## Introduction

No real human preference data exists for this notebook to train on,
so — exactly like 13a's synthetic Bradley-Terry comparisons — a simple,
fully specified stand-in criterion plays the role real human judgment
would: **vowel density** (the fraction of a completion's characters that
are vowels). It is a deliberately crude, one-dimensional proxy for
"preferred text," chosen precisely *because* it is crude: a proxy this
simple is easy to game, which is exactly what the "Failure Modes"
section needs to demonstrate honestly, on a real model, rather than only
describe in the abstract.

## Setup

In [ ]:
# Fixed seeds: every stochastic step (weight init, sampling, preference
# labelling) is reproducible.
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)

# Same corpus as 7b/10b.
paragraph_1 = [
    "It is a truth universally acknowledged, that a single man in possession ",
    "of a good fortune, must be in want of a wife. However little known the ",
    "feelings or views of such a man may be on his first entering a ",
    "neighbourhood, this truth is so well fixed in the minds of the ",
    "surrounding families, that he is considered as the rightful property of ",
    "some one or other of their daughters. ",
]
paragraph_2 = [
    '"My dear Mr. Bennet," said his lady to him one day, "have you heard ',
    'that Netherfield Park is let at last?" ',
    "Mr. Bennet replied that he had not. ",
    '"But it is," returned she; "for Mrs. Long has just been here, and she ',
    "told me all about it.\" ",
    "Mr. Bennet made no answer. ",
    '"Do not you want to know who has taken it?" cried his wife impatiently. ',
    '"You want to tell me, and I have no objection to hearing it." ',
]
paragraph_3 = [
    "This was invitation enough. ",
    '"Why, my dear, you must know, Mrs. Long says that Netherfield is taken ',
    "by a young man of large fortune from the north of England; that he came ",
    "down on Monday in a chaise and four to see the place, and was so much ",
    "delighted with it, that he agreed with Mr. Morris immediately; that he ",
    "is to take possession before Michaelmas, and some of his servants are ",
    "to be in the house by the end of next week.\" ",
    '"What is his name?" ',
    '"Bingley." ',
    '"Is he married or single?" ',
    '"Oh! Single, my dear, to be sure! A single man of large fortune; four ',
    "or five thousand a year. What a fine thing for our girls!\" ",
]

CORPUS = "".join(paragraph_1 + paragraph_2 + paragraph_3)
chars = sorted(set(CORPUS))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return torch.tensor([stoi[c] for c in s], dtype=torch.long)

train_data = encode(CORPUS)
print(f"corpus: {len(CORPUS)} chars, vocab: {vocab_size}")

def vowel_density(text):
    return sum(c.lower() in "aeiou" for c in text) / max(len(text), 1)

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=32, hidden_dim=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        out, hidden = self.lstm(self.embed(x), hidden)
        return self.fc(out), hidden


def get_batch(data, seq_len, batch_size, rng):
    max_start = len(data) - seq_len - 1
    starts = rng.integers(0, max_start, size=batch_size)
    x = torch.stack([data[s:s + seq_len] for s in starts])
    y = torch.stack([data[s + 1:s + seq_len + 1] for s in starts])
    return x, y


def pretrain(model, n_iters=400, batch_size=32, seq_len=40, lr=2e-3, seed=SEED):
    rng = np.random.default_rng(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(n_iters):
        xb, yb = get_batch(train_data, seq_len, batch_size, rng)
        logits, _ = model(xb)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.35)
        optimizer.step()
    return loss.item()


def sample(model, seed_text, n_chars, temperature, rng):
    model.eval()
    idxs = [stoi[c] for c in seed_text]
    x = torch.tensor(idxs, dtype=torch.long).unsqueeze(0)
    with torch.no_grad():
        logits, hidden = model(x)
    out = list(seed_text)
    next_input = x[:, -1:]
    with torch.no_grad():
        for _ in range(n_chars):
            logits, hidden = model(next_input, hidden)
            probs = F.softmax(logits[0, -1] / temperature, dim=-1).numpy()
            ix = rng.choice(vocab_size, p=probs)
            out.append(itos[ix])
            next_input = torch.tensor([[ix]], dtype=torch.long)
    model.train()
    return "".join(out[len(seed_text):])  # completion only, not the prompt


torch.manual_seed(SEED)
pi_ref = CharLSTM(vocab_size)
train_loss = pretrain(pi_ref)
print(f"pretraining final loss: {train_loss:.3f}")

## Training a Reward Model

A tiny preference set is built by sampling several completions per
prompt from the pretrained policy, then labelling each pair
stochastically with the Bradley-Terry model itself, using vowel density
as the underlying "true" reward — the exact same construction 13a used
for a synthetic ranking problem, now applied to real generated text.

In [ ]:
PROMPTS = ["It is a ", "Mr. Benn", "the surro", "she said ", "the house", "a young m"]
COMPLETION_LEN = 40
N_COMPLETIONS_PER_PROMPT = 8
BT_SCALE = 20.0  # sharpens noisy vowel-density differences into learnable labels

sample_rng = np.random.default_rng(SEED)
completions_by_prompt = {
    p: [sample(pi_ref, p, COMPLETION_LEN, temperature=0.9, rng=sample_rng)
        for _ in range(N_COMPLETIONS_PER_PROMPT)]
    for p in PROMPTS
}

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

pref_rng = np.random.default_rng(SEED)
preference_pairs = []  # (prompt, winner_text, loser_text)
for prompt, completions in completions_by_prompt.items():
    for i in range(len(completions)):
        for j in range(i + 1, len(completions)):
            r_i, r_j = vowel_density(completions[i]), vowel_density(completions[j])
            p_i_wins = sigmoid(BT_SCALE * (r_i - r_j))
            i_wins = pref_rng.random() < p_i_wins
            winner, loser = (completions[i], completions[j]) if i_wins else (completions[j], completions[i])
            preference_pairs.append((prompt, winner, loser))

pref_rng.shuffle(preference_pairs)
split = int(0.8 * len(preference_pairs))
train_pairs, held_out_pairs = preference_pairs[:split], preference_pairs[split:]
print(f"{len(preference_pairs)} preference pairs ({len(train_pairs)} train, {len(held_out_pairs)} held out)")
print(f"example: prompt={train_pairs[0][0]!r}")
print(f"  preferred: {train_pairs[0][1]!r} (vowel density {vowel_density(train_pairs[0][1]):.2f})")
print(f"  rejected:  {train_pairs[0][2]!r} (vowel density {vowel_density(train_pairs[0][2]):.2f})")

In [ ]:
class RewardModel(nn.Module):
    def __init__(self, vocab_size, emb_dim=16, hidden_dim=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, text):
        x = encode(text).unsqueeze(0)
        _, (h, _) = self.lstm(self.embed(x))
        return self.head(h.squeeze(0)).squeeze()


torch.manual_seed(SEED)
reward_model = RewardModel(vocab_size)
optimizer = torch.optim.Adam(reward_model.parameters(), lr=1e-3)

reward_losses = []
for epoch in range(30):
    epoch_loss = 0.0
    for prompt, winner, loser in train_pairs:
        r_w = reward_model(winner)
        r_l = reward_model(loser)
        loss = -F.logsigmoid(r_w - r_l)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    reward_losses.append(epoch_loss / len(train_pairs))

with torch.no_grad():
    correct = sum(reward_model(w).item() > reward_model(l).item() for _, w, l in held_out_pairs)
pairwise_accuracy = correct / len(held_out_pairs)
print(f"reward model training loss: {reward_losses[0]:.3f} -> {reward_losses[-1]:.3f}")
print(f"held-out pairwise accuracy: {pairwise_accuracy:.1%} ({correct}/{len(held_out_pairs)})")

The reward model was never told what vowel density is — it only ever
saw which of two completions a human (here, the synthetic Bradley-Terry
label) preferred — and it recovers the underlying preference modestly
but genuinely above chance (50%) on preference pairs it never trained
on, from only 134 training comparisons over completions whose vowel
density barely varies (roughly 0.2–0.4 across the whole sampled set).
Real reward models train on vastly more comparisons over far more
varied text; this is the same mechanism at a scale small enough to run
in seconds.

## Running DPO

13a's DPO loss generalises to full sequences directly: $\log
\pi_\theta(y \mid x)$ for a whole completion $y$ is just the sum of its
per-token teacher-forced log-probabilities under the model, computed
once per (prompt, completion) pair — no sampling inside the training
loop, no reward model, exactly 13a's derivation applied to sequences
instead of single logits:

$$\mathcal{L}_{\text{DPO}} = -\log\sigma\!\Big(\beta\big[(\log\pi_\theta(y_w|x) - \log\pi_{\text{ref}}(y_w|x)) - (\log\pi_\theta(y_l|x) - \log\pi_{\text{ref}}(y_l|x))\big]\Big).$$

$\pi_{\text{ref}}$ is a frozen copy of the pretrained model; only
$\pi_\theta$'s weights are updated, directly on the preference pairs
built above — the reward model trained in the previous section is not
used here at all, which is precisely DPO's point.

In [ ]:
def sequence_logprob(model, prompt, completion):
    full = prompt + completion
    x = encode(full[:-1]).unsqueeze(0)
    targets = encode(full[1:]).unsqueeze(0)
    logits, _ = model(x)
    log_probs = F.log_softmax(logits, dim=-1)
    token_logprobs = log_probs.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    completion_start = len(prompt) - 1  # predictions covering the completion's tokens
    return token_logprobs[0, completion_start:].sum()


pi_theta = copy.deepcopy(pi_ref)
for p in pi_ref.parameters():
    p.requires_grad_(False)

BETA = 0.05


def dpo_train(model, optimizer, pairs, n_epochs, beta=BETA):
    losses = []
    for epoch in range(n_epochs):
        epoch_loss = 0.0
        for prompt, winner, loser in pairs:
            logp_w_theta = sequence_logprob(model, prompt, winner)
            logp_l_theta = sequence_logprob(model, prompt, loser)
            with torch.no_grad():
                logp_w_ref = sequence_logprob(pi_ref, prompt, winner)
                logp_l_ref = sequence_logprob(pi_ref, prompt, loser)
            z = beta * ((logp_w_theta - logp_w_ref) - (logp_l_theta - logp_l_ref))
            loss = -F.logsigmoid(z)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(pairs))
    return losses


dpo_optimizer = torch.optim.Adam(pi_theta.parameters(), lr=5e-4)
dpo_losses = dpo_train(pi_theta, dpo_optimizer, train_pairs, n_epochs=8)
print(f"DPO loss: {dpo_losses[0]:.3f} -> {dpo_losses[-1]:.3f}")

In [ ]:
plt.figure()
plt.plot(dpo_losses)
plt.xlabel("epoch")
plt.ylabel("mean DPO loss")
plt.title("DPO training directly on preference pairs")
plt.tight_layout()
plt.show()

## Before and After

Generating from `pi_ref` and `pi_theta` on prompts *never used to
build the preference set* isolates whether DPO changed general
behaviour, not just memorised the training completions.

In [ ]:
held_out_prompts = ["and Mrs. ", "his wife "]
eval_rng = np.random.default_rng(SEED)

for prompt in held_out_prompts:
    before = sample(pi_ref, prompt, COMPLETION_LEN, temperature=0.8, rng=eval_rng)
    after = sample(pi_theta, prompt, COMPLETION_LEN, temperature=0.8, rng=eval_rng)
    print(f"prompt: {prompt!r}")
    print(f"  before DPO: {before!r}  (vowel density {vowel_density(before):.2f})")
    print(f"  after DPO:  {after!r}  (vowel density {vowel_density(after):.2f})")
    print()

On prompts the preference set never contained, the post-DPO model's
vowel density moves modestly higher than the pretrained model's on both
held-out prompts — a small but consistent shift, not a dramatic one, from
only 8 epochs over 134 preference pairs. DPO's update was driven purely
by the training prompts' preference pairs, and that small a nudge is
already enough to generalise in the right direction to prompts it never
trained on; "Failure Modes" next shows what happens when the same
mechanism is pushed much harder.

## Failure Modes

**Reward hacking** (or "over-optimisation") is what happens when a
policy is pushed to maximise a *proxy* for what is actually wanted, past
the point where the proxy and the real goal agree. Vowel density is a
transparent example precisely because its failure mode is so easy to
produce and recognise: a model can trivially maximise it by degenerating
into near-nonsense vowel-heavy output, which is exactly what real reward
hacking does to a real, more subtle reward model — the proxy keeps going
up while the thing it was supposed to measure gets worse.

In [ ]:
# A much weaker KL anchor (smaller beta) and a larger learning rate --
# starting fresh from pi_ref, not continuing from the well-behaved run
# above -- lets the same preference pairs push far harder per step.
pi_theta_overtrained = copy.deepcopy(pi_ref)
for p in pi_theta_overtrained.parameters():
    p.requires_grad_(True)
overtrain_optimizer = torch.optim.Adam(pi_theta_overtrained.parameters(), lr=2e-3)
_ = dpo_train(pi_theta_overtrained, overtrain_optimizer, train_pairs, n_epochs=40, beta=0.01)

overtrain_rng = np.random.default_rng(SEED)
for prompt in held_out_prompts:
    over_generated = sample(pi_theta_overtrained, prompt, COMPLETION_LEN, temperature=0.8, rng=overtrain_rng)
    print(f"prompt: {prompt!r}")
    print(f"  weak-KL-anchor DPO: {over_generated!r}  (vowel density {vowel_density(over_generated):.2f})")

With a much weaker KL anchor ($\beta = 0.01$ instead of $0.05$) and a
larger learning rate — the same, never-enlarged preference set, only a
far weaker penalty for drifting from $\pi_{\text{ref}}$ — the model
collapses into repeating short vowel-heavy fragments almost verbatim,
rather than producing anything resembling English. Vowel density does
rise, exactly as the proxy rewards, but the text that achieves it is
obviously worse by the only measure that actually matters: whether it
still says anything. The model found the cheapest way to satisfy the
proxy rather than genuinely writing better sentences, because nothing in
the DPO objective asks for coherence — only for the preferred completion
to keep beating the rejected one, and a weak enough KL anchor lets the
policy pay almost any coherence price to do that. Real RLHF pipelines
guard against exactly this with a properly tuned KL penalty against
$\pi_{\text{ref}}$ (13a) and by stopping training once held-out
preference accuracy or a separate quality check stops improving; this
demonstration deliberately weakens that anchor instead, so the failure
mode is visible rather than engineered away.

## Key Takeaways

- **A reward model trained on a tiny synthetic preference set reached
  modest but genuine above-chance pairwise accuracy on held-out
  comparisons** it never trained on, recovering an underlying preference
  criterion it was never told directly, from as few as 134 comparisons.
- **DPO fine-tuned a real character-level language model's own weights
  directly on preference pairs**, with no reward model or sampling
  inside the training loop, and even a small, well-anchored update
  generalised in the right direction to prompts outside the preference
  set.
- **Weakening the KL anchor against a crude proxy reward reproduces
  reward hacking directly, not just in description**: the same
  preference pairs, optimised with a much smaller $\beta$, drove the
  proxy (vowel density) up while the generated text collapsed into
  repetitive near-nonsense — evidence that alignment training needs a
  properly weighted anchor and a check on genuine quality, not only on
  the proxy it is directly optimising.